# Master's Thesis: CNN–LSTM Network Intrusion Detection
## Dual-Head Anomaly Detection for Zero-Day and Low-Frequency Intrusions
**Datasets**: CICIDS2017 & CSE-CIC-IDS2018 (18.35 Million Flows)
**Target Hardware**: Free Cloud GPU (Google Colab NVIDIA T4 / Kaggle P100)

---
### 1-Click Execution Instructions:
1. **Enable Free GPU**: Go to `Runtime` -> `Change runtime type` -> select `T4 GPU` -> Click `Save`.
2. **Upload `NID Project.rar`**: Simply place `NID Project.rar` in your Google Drive.
3. **Click `Runtime -> Run all`**: The notebook will automatically extract `NID Project.rar`, train the model on the T4 GPU, evaluate zero-day detection, and generate your thesis reports!

### Step 1: Verify NVIDIA GPU Accelerator

In [ ]:
!nvidia-smi
import torch
print('=' * 60)
print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name :', torch.cuda.get_device_name(0))
    print('VRAM Available  :', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')
else:
    print('WARNING: GPU is not active. Go to Runtime -> Change runtime type -> T4 GPU.')
print('=' * 60)

### Step 2: Mount Google Drive & Auto-Extract `NID Project.rar`
If you uploaded `NID Project.rar` to Google Drive, this cell automatically extracts it to Colab's high-speed local NVMe SSD (`/content/NID Project`).

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# 1. Check for NID Project.rar in Google Drive
rar_candidates = [
    '/content/drive/MyDrive/NID Project.rar',
    '/content/drive/MyDrive/NID_Project.rar',
]
rar_path = next((p for p in rar_candidates if os.path.exists(p)), None)

if rar_path:
    print(f'[OK] Found archive: {rar_path}')
    if not os.path.exists('/content/NID Project'):
        print('Extracting to /content/ ... (takes ~30-45 seconds)')
        !unrar x -inul "{rar_path}" /content/
        print('Extraction complete!')
    os.chdir('/content/NID Project')
elif os.path.exists('/content/drive/MyDrive/NID Project'):
    os.chdir('/content/drive/MyDrive/NID Project')
    print('Navigated to Drive folder: /content/drive/MyDrive/NID Project')
else:
    print('Notice: NID Project.rar not found in root of Drive.')
    print('Contents of MyDrive:', os.listdir('/content/drive/MyDrive')[:10])

print('Working Directory:', os.getcwd())
!ls -la

### Step 3: Install Required Dependencies

In [ ]:
!pip install --quiet torch torchvision scikit-learn pandas numpy joblib

### Step 4: Train Dual-Head CNN–LSTM on Cloud GPU
Trains the full 1D-CNN + BiLSTM model across all 12.84 million training flows using CUDA acceleration. Each epoch takes only **~5 to 6 minutes** on an NVIDIA T4 GPU.

In [ ]:
# Full multi-epoch GPU training
!python scripts/train_cnn_lstm.py --epochs 3 --batch_size 512 --seq_len 10 --device cuda

### Step 5: Benchmark Evaluation on Known & Zero-Day Intrusions
Evaluates the trained checkpoint on:
1. **Validation Traffic**: Calibrates decision threshold $\tau$ at 1.0% False Positive Rate.
2. **Known Attacks Test**: Measures Macro-F1 across 12 known intrusion classes.
3. **Zero-Day Benchmark**: Tests anomaly detection recall on the 1,001 unseen flows (**Heartbleed**, **SQL Injection**, **Brute Force - XSS**).

In [ ]:
!python scripts/evaluate_zero_day.py --batch_size 512 --seq_len 10 --target_fpr 0.01 --device cuda

### Step 6: Display Human-Level Performance Interpretation Report

In [ ]:
from IPython.display import display, Markdown
import os

interp_md = 'results/Model_Performance_Interpretation_Report.md'
if os.path.exists(interp_md):
    with open(interp_md, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print('Report not found. Please run Step 5 first.')

### Step 7: Download Model Checkpoints & Reports to Local PC
Download `best_cnn_lstm.pt` and all generated evaluation reports back to your local computer.

In [ ]:
from google.colab import files

# Download trained checkpoint
if os.path.exists('models/best_cnn_lstm.pt'):
    print('Downloading best_cnn_lstm.pt...')
    files.download('models/best_cnn_lstm.pt')

# Download evaluation artifacts & human interpretation report
for fname in [
    'results/zero_day_evaluation_results.json',
        'results/low_frequency_attacks_report.json',
    'results/known_attacks_classification_report.csv',
    'results/Model_Performance_Interpretation_Report.md',
]:
    if os.path.exists(fname):
        print(f'Downloading {fname}...')
